# Semana 14: Despliegue de Modelos NLP (MLOps)
## Notebook Conceptual (NB1) – API con FastAPI y Docker

**Propósito:** Llevar un modelo NLP a producción: serialización, creación de API con FastAPI, contenerización con Docker y pruebas locales.

**Docente:** Carlos César Sánchez Coronel

**Objetivos de aprendizaje:**
- Serializar un modelo fine-tuneado con `torch.save` y el tokenizer con `save_pretrained`.
- Crear una API REST con FastAPI con un endpoint `/predict`.
- Probar la API localmente con `requests`.
- Escribir un Dockerfile y construir la imagen.
- Ejecutar el contenedor y probar el endpoint.

---

## 0. Configuración Inicial

Importamos las librerías necesarias e instalamos dependencias.

In [ ]:
# Instalamos dependencias
!pip install fastapi uvicorn transformers torch requests nest-asyncio pyngrok

# Importamos librerías
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import torch
import time
import requests
import json
import os
import subprocess
import sys
import nest_asyncio
import threading
import warnings
warnings.filterwarnings('ignore')

# Transformers
from transformers import AutoTokenizer, AutoModelForSequenceClassification

# FastAPI y uvicorn
from fastapi import FastAPI
from pydantic import BaseModel
import uvicorn

# Para ngrok (opcional, para exponer la API)
from pyngrok import ngrok

# Configuración de visualización
%matplotlib inline
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (12, 6)

# Verificar dispositivo
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Usando dispositivo: {device}")

print("\nLibrerías importadas correctamente.")

---
## 1. Modelo de Ejemplo (Clasificador de Sentimiento)

Para este ejercicio, usaremos un modelo BERT fine-tuneado para clasificación de sentimiento. Si no tenemos uno guardado, cargamos un modelo pre-entrenado de Hugging Face.

In [ ]:
# Cargar modelo y tokenizer
model_name = "distilbert-base-uncased-finetuned-sst-2-english"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(model_name)

# Mover modelo a dispositivo
model.to(device)
model.eval()

print("Modelo de ejemplo cargado.")
print(f"Clases: {model.config.id2label}")

---
## 2. Serialización del Modelo

Guardamos el modelo y el tokenizer en disco para usarlos en la API.

In [ ]:
# Crear directorio para el modelo
model_dir = "./sentiment_model"
os.makedirs(model_dir, exist_ok=True)

# Guardar tokenizer
tokenizer.save_pretrained(model_dir)

# Guardar modelo con torch.save
torch.save(model.state_dict(), os.path.join(model_dir, "pytorch_model.bin"))

# También guardamos la configuración
model.config.save_pretrained(model_dir)

print(f"Modelo y tokenizer guardados en {model_dir}")
!ls -la {model_dir}

---
## 3. Creación de la API con FastAPI

Creamos un archivo `app.py` que contendrá nuestra API.

In [ ]:
app_content = '''
import torch
from fastapi import FastAPI
from pydantic import BaseModel
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import os

# Configuración
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model_dir = "./sentiment_model"

# Cargar modelo y tokenizer
tokenizer = AutoTokenizer.from_pretrained(model_dir)
model = AutoModelForSequenceClassification.from_pretrained(model_dir)
model.to(device)
model.eval()

# Definir esquema de entrada
class TextInput(BaseModel):
    text: str

# Crear app
app = FastAPI(title="Sentiment Analysis API", description="API para clasificación de sentimiento", version="1.0")

@app.get("/")
def root():
    return {"message": "Sentiment Analysis API. Use /predict con POST."}

@app.post("/predict")
def predict(input_data: TextInput):
    try:
        # Tokenizar
        inputs = tokenizer(input_data.text, return_tensors="pt", truncation=True, padding=True, max_length=128)
        inputs = {k: v.to(device) for k, v in inputs.items()}
        
        # Inferencia
        with torch.no_grad():
            outputs = model(**inputs)
            logits = outputs.logits
            probabilities = torch.softmax(logits, dim=-1)
            prediction = torch.argmax(logits, dim=-1).item()
        
        # Mapear etiquetas
        id2label = model.config.id2label
        label = id2label[prediction]
        confidence = probabilities[0][prediction].item()
        
        return {
            "prediction": label,
            "confidence": confidence,
            "probabilities": probabilities[0].tolist()
        }
    except Exception as e:
        return {"error": str(e)}

if __name__ == "__main__":
    import uvicorn
    uvicorn.run(app, host="0.0.0.0", port=8000)
'''

with open("app.py", "w") as f:
    f.write(app_content)

print("Archivo app.py creado.")

---
## 4. Archivo de Requisitos

Creamos `requirements.txt` con las dependencias necesarias.

In [ ]:
requirements_content = '''
torch
transformers
fastapi
uvicorn
pydantic
numpy
'''

with open("requirements.txt", "w") as f:
    f.write(requirements_content)

print("Archivo requirements.txt creado.")

---
## 5. Ejecutar la API Localmente

Ejecutamos la API en un proceso separado para poder probarla desde el notebook.

In [ ]:
# Función para ejecutar la API en segundo plano
def run_api():
    os.system("uvicorn app:app --host 0.0.0.0 --port 8000 --reload")

# Ejecutar en un hilo separado
api_thread = threading.Thread(target=run_api, daemon=True)
api_thread.start()

# Esperar a que la API inicie
time.sleep(5)
print("API iniciada en http://localhost:8000")

---
## 6. Probar la API con Requests

Enviamos peticiones al endpoint `/predict` para verificar que funciona.

In [ ]:
def test_api(text):
    url = "http://localhost:8000/predict"
    payload = {"text": text}
    headers = {"Content-Type": "application/json"}
    
    try:
        response = requests.post(url, json=payload, headers=headers)
        if response.status_code == 200:
            return response.json()
        else:
            return {"error": f"Status code: {response.status_code}", "detail": response.text}
    except Exception as e:
        return {"error": str(e)}

test_texts = [
    "I absolutely loved this movie, it was fantastic!",
    "This film was terrible and boring, complete waste of time.",
    "It was okay, not great but not bad either."
]

print("=== PRUEBA DE LA API ===\n")
for text in test_texts:
    result = test_api(text)
    print(f"Texto: {text[:50]}...")
    if 'prediction' in result:
        print(f"Predicción: {result['prediction']}")
        print(f"Confianza: {result['confidence']:.4f}")
    else:
        print(f"Error: {result.get('error', 'Desconocido')}")
    print("-"*50)

### 6.1. (Opcional) Exponer la API con ngrok

Si queremos probar la API desde internet, podemos usar ngrok.

In [ ]:
# Configurar ngrok (necesitas un token de autenticación)
# Descomentar y configurar tu token si deseas usar ngrok
# ngrok.set_auth_token("TU_TOKEN")
# public_url = ngrok.connect(8000)
# print(f"API pública en: {public_url}")

print("Para usar ngrok, descomenta las líneas y configura tu token.")

---
## 7. Creación del Dockerfile

Creamos un Dockerfile para contenerizar la API.

In [ ]:
dockerfile_content = '''
FROM python:3.9-slim

WORKDIR /app

# Copiar archivos de requisitos
COPY requirements.txt .

# Instalar dependencias
RUN pip install --no-cache-dir -r requirements.txt

# Copiar modelo y código
COPY sentiment_model/ ./sentiment_model/
COPY app.py .

# Exponer puerto
EXPOSE 8000

# Comando para ejecutar la API
CMD ["uvicorn", "app:app", "--host", "0.0.0.0", "--port", "8000"]
'''

with open("Dockerfile", "w") as f:
    f.write(dockerfile_content)

print("Archivo Dockerfile creado.")
!cat Dockerfile

### 7.1. Archivo .dockerignore

Creamos un `.dockerignore` para evitar copiar archivos innecesarios.

In [ ]:
dockerignore_content = '''
__pycache__
*.pyc
.git
.env
notebooks
data
'''

with open(".dockerignore", "w") as f:
    f.write(dockerignore_content)

print("Archivo .dockerignore creado.")

---
## 8. Construir y Ejecutar el Contenedor Docker

Construimos la imagen Docker y ejecutamos el contenedor.

**Nota**: Estos comandos deben ejecutarse en una terminal con Docker instalado. En Colab no podemos ejecutar Docker directamente, pero se muestran los comandos para referencia.

In [ ]:
print("=== COMANDOS DOCKER ===")
print("\n1. Construir la imagen:")
print("   docker build -t sentiment-api .")
print("\n2. Ejecutar el contenedor:")
print("   docker run -p 8000:8000 sentiment-api")
print("\n3. Ver contenedores en ejecución:")
print("   docker ps")
print("\n4. Detener el contenedor:")
print("   docker stop <container_id>")

### 8.1. (Opcional) Docker Compose

Para sistemas más complejos, podemos usar Docker Compose.

In [ ]:
compose_content = '''
version: '3.8'

services:
  api:
    build: .
    ports:
      - "8000:8000"
    volumes:
      - ./sentiment_model:/app/sentiment_model
    environment:
      - CUDA_VISIBLE_DEVICES=0
'''

with open("docker-compose.yml", "w") as f:
    f.write(compose_content)

print("Archivo docker-compose.yml creado.")

---
## 9. Medición de Latencia y Throughput

Podemos medir la latencia de nuestra API.

In [ ]:
def measure_latency(n_requests=10):
    """
    Mide la latencia de la API.
    """
    test_text = "I really enjoyed this movie, it was great!"
    url = "http://localhost:8000/predict"
    payload = {"text": test_text}
    
    latencies = []
    for i in range(n_requests):
        start = time.time()
        response = requests.post(url, json=payload)
        end = time.time()
        if response.status_code == 200:
            latencies.append((end - start) * 1000)  # en ms
    
    return latencies

latencies = measure_latency(10)
if latencies:
    print(f"Latencia media: {np.mean(latencies):.2f} ms")
    print(f"Desviación estándar: {np.std(latencies):.2f} ms")
    print(f"Mínimo: {min(latencies):.2f} ms")
    print(f"Máximo: {max(latencies):.2f} ms")
    
    plt.figure(figsize=(10, 4))
    plt.plot(latencies, 'o-')
    plt.xlabel('Solicitud')
    plt.ylabel('Latencia (ms)')
    plt.title('Latencia de la API')
    plt.grid(True)
    plt.show()
else:
    print("No se pudieron medir latencias. ¿La API está corriendo?")

---
## 10. Limpieza

Detenemos la API (en Colab, el hilo se detendrá al finalizar el notebook).

In [ ]:
# El hilo daemon se detendrá automáticamente al finalizar el notebook
print("API detenida.")

---
## 11. Ejercicio para el Estudiante

1. **Modifica la API**: Añade un endpoint `/batch_predict` que acepte una lista de textos.

2. **Mejora el Dockerfile**: Añade una etapa de build con dependencias de CUDA si usas GPU.

3. **Despliega en la nube**: Prueba a desplegar la imagen en un servicio como AWS ECS, Google Cloud Run o Azure.

4. **Añade autenticación**: Implementa una clave API simple en FastAPI.

5. **Monitoreo**: Añade logging de métricas (latencia, errores) a un archivo.

In [ ]:
# Espacio para el estudiante
# ...

---
## 12. Conclusiones

En este notebook hemos desplegado un modelo NLP en producción:

✔️ **Serialización**: Guardamos modelo y tokenizer con `torch.save` y `save_pretrained`.

✔️ **API REST**: Creamos una API con FastAPI con endpoint `/predict`.

✔️ **Pruebas locales**: Verificamos el funcionamiento con `requests`.

✔️ **Contenerización**: Escribimos un Dockerfile para empaquetar la aplicación.

✔️ **Docker Compose**: Opcionalmente, para sistemas más complejos.

✔️ **Medición de latencia**: Evaluamos el rendimiento de la API.

**Lección clave**: El despliegue de modelos NLP en producción requiere serialización, una API robusta y contenerización para garantizar reproducibilidad y escalabilidad.

---
**Fin del Notebook Conceptual - Semana 14 (NLP)**
**Fin del Curso de Procesamiento de Lenguaje Natural**